# Подготовка адресов для НСПД

Ноутбук читает `полный_датасет.csv`, берёт заполненные значения `full_address`, убирает повторы и сохраняет отдельный файл `адреса_для_nspd.csv`.

В выходном файле находится только одна колонка — `full_address`.


In [ ]:
from pathlib import Path

import pandas as pd


## 1. Файлы


In [ ]:
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / 'результаты').exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / 'результаты').exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'парсинги' else CURRENT_DIR

RESULTS_DIR = PROJECT_ROOT / 'результаты'
INPUT_FILE = RESULTS_DIR / 'полный_датасет.csv'
OUTPUT_FILE = RESULTS_DIR / 'адреса_для_nspd.csv'

print('Входной файл:', INPUT_FILE)
print('Выходной файл:', OUTPUT_FILE)


## 2. Чтение полного датасета


In [ ]:
def read_csv_flexible(path):
    last_error = None
    for encoding in ['utf-8-sig', 'utf-8', 'cp1251']:
        try:
            return pd.read_csv(
                path,
                sep=None,
                engine='python',
                encoding=encoding,
                dtype='string',
            )
        except Exception as error:
            last_error = error
    raise RuntimeError(f'Не удалось прочитать {path}: {last_error}')


if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Не найден входной файл: {INPUT_FILE}')

full_dataset = read_csv_flexible(INPUT_FILE)
full_dataset.columns = [str(column).strip() for column in full_dataset.columns]

if 'full_address' not in full_dataset.columns:
    raise ValueError('В полном датасете нет колонки full_address')

print('Строк в полном датасете:', len(full_dataset))
print('Колонок в полном датасете:', len(full_dataset.columns))


## 3. Выгрузка уникальных адресов


In [ ]:
clean_address = full_dataset['full_address'].astype('string').str.strip()
clean_address = clean_address.mask(
    clean_address.eq('')
    | clean_address.str.lower().isin(['nan', 'none', 'null'])
)

addresses = pd.DataFrame({'full_address': clean_address})
addresses = (
    addresses
    .dropna(subset=['full_address'])
    .drop_duplicates(subset=['full_address'])
    .sort_values('full_address')
    .reset_index(drop=True)
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
addresses.to_csv(
    OUTPUT_FILE,
    index=False,
    sep=';',
    encoding='utf-8-sig',
    lineterminator='\n',
)

print('Строк с заполненным адресом:', int(clean_address.notna().sum()))
print('Уникальных адресов:', len(addresses))
print('Файл сохранён:', OUTPUT_FILE)


## 4. Проверка созданного файла


In [ ]:
addresses_from_file = pd.read_csv(
    OUTPUT_FILE,
    sep=';',
    encoding='utf-8-sig',
    dtype='string',
)

if list(addresses_from_file.columns) != ['full_address']:
    raise ValueError('В файле должна быть только колонка full_address')
if addresses_from_file['full_address'].isna().any():
    raise ValueError('В файл попали пустые адреса')
if addresses_from_file['full_address'].duplicated().any():
    raise ValueError('В файл попали повторяющиеся адреса')

print('Файл успешно прочитан')
print('Адресов в файле:', len(addresses_from_file))
display(addresses_from_file.head(10))
